# LiDAR 3D Object Detection — PointPillars on Waymo Open Dataset

Trains and evaluates a **PointPillars** 3-D object detector on LiDAR data
streamed live from GCS.  **No local dataset download required.**

### Architecture

| Stage | Module | Output shape |
|---|---|---|
| Voxelization | `_voxelize()` | `(P, 20, 9)` |
| Pillar feature net | `PillarFeatureNet` (PointNet MLP + max-pool) | `(P, 64)` |
| Pillar scatter | `PillarScatter` | `(B, 64, 400, 400)` |
| BEV backbone | 2-stage 2-D CNN (64→128) | `(B, 128, 200, 200)` |
| Detection head | heatmap + offset + Z + log-dim + heading | per-class `(B, 200, 200)` |

**Loss:** Gaussian focal loss (heatmap) + L1 (box regression)  
**Inference:** heatmap peak detection + box decoding  *(no NMS needed)*

### Sections
1. [Setup & auth](#cell-setup)
2. [Configuration](#cell-config)
3. [Pre-flight check](#cell-preflight) ← verifies GCS column names before training
4. [LiDAR frame preview (BEV)](#cell-preview)
5. [Training](#cell-train)
6. [Load checkpoint & inference](#cell-load)
7. [BEV visualisation](#cell-bev)
8. [Evaluation (BEV mAP)](#cell-eval)


In [ ]:
# §1 — Setup & GCS auth
import os, subprocess, sys

# ── Google Drive (checkpoint storage) ────────────────────────────────────────
try:
    from google.colab import drive, auth
    auth.authenticate_user()
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Colab: Drive mounted, GCS credentials set.')
except ModuleNotFoundError:
    IN_COLAB = False
    print('Local mode — Drive not mounted.  Run: gcloud auth application-default login')

# ── Clone / pull repo ─────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/KushalBKusram/WaymoOpenDatasetToolKit.git'
REPO_DIR = '/content/WaymoOpenDatasetToolKit' if IN_COLAB else '.'

if IN_COLAB and not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
elif IN_COLAB:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Install dependencies ───────────────────────────────────────────────────────
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', 'requirements.txt'], check=True)
    print('Dependencies installed.')

# ── GCS connectivity check ─────────────────────────────────────────────────────
import tensorflow as tf
test_files = tf.io.gfile.glob(
    'gs://waymo_open_dataset_v_2_0_0/training/lidar/*.parquet'
)
print(f'GCS auth OK — {len(test_files)} training LiDAR Parquet files found.')


In [ ]:
# §2 — Configuration
import yaml
from pathlib import Path

# ── Edit these two lines ───────────────────────────────────────────────────────
CONFIG    = 'configs/pointpillars.yaml'
DRIVE_DIR = '/content/drive/MyDrive/waymo_lidar'   # checkpoint destination

# ── Optional one-off overrides (None = use YAML value) ────────────────────────
TOTAL_SEGS_OVERRIDE = None   # e.g. 3 for a quick smoke test
BATCH_OVERRIDE      = None   # e.g. 1 if OOM

with open(CONFIG) as f:
    active_cfg = yaml.safe_load(f)

vox = active_cfg['voxel']
print(f"Config        : {active_cfg['name']}")
print(f"Task          : {active_cfg['task']}")
print(f"Detector      : {active_cfg['model']['type']}")
print(f"Classes       : {active_cfg['model']['class_names']}")
print(f"Voxel X range : {vox['x_range']}  ({(vox['x_range'][1]-vox['x_range'][0]):.0f} m)")
print(f"Voxel Y range : {vox['y_range']}  ({(vox['y_range'][1]-vox['y_range'][0]):.0f} m)")
print(f"Pillar size   : {vox['voxel_size']} m  →  "
      f"{round((vox['x_range'][1]-vox['x_range'][0])/vox['voxel_size'][0])}\u00d7"
      f"{round((vox['y_range'][1]-vox['y_range'][0])/vox['voxel_size'][1])} BEV")
print(f"Max pillars   : {vox['max_pillars']}")
print(f"Max pts/pillar: {vox['max_points_per_pillar']}")
print(f"Batch size    : {active_cfg['train']['batch_size']}")
print(f"LR            : {active_cfg['train']['lr']}")
print(f"Total segs    : {TOTAL_SEGS_OVERRIDE or active_cfg['train']['total_segs']}")
print(f"Drive dir     : {DRIVE_DIR}")


In [ ]:
# §3 — Pre-flight check
#
# Reads ONE Parquet file from GCS and verifies the column names we rely on
# exist before wasting time on a full training run.
# Also initialises `tk` and `segs` used by every subsequent cell.

import numpy as np
import matplotlib.pyplot as plt
from modules.waymo_open_dataset import ToolKit, LIDAR_DET_CLASS_MAP, _L_BOX

# ── Initialise toolkit (shared across all cells) ───────────────────────────────
tk   = ToolKit(split=active_cfg['data']['split'])
segs = tk.list_segments()
print(f'{len(segs)} segments found in split="{active_cfg["data"]["split"]}"')

tk.assign_segment(segs[0])

# ── Print actual lidar_box column names ────────────────────────────────────────
print('\n── lidar_box Parquet columns ─────────────────────────────────────')
tk.debug_columns('lidar_box')

# ── Verify the columns we rely on exist ───────────────────────────────────────
REQUIRED_LIDAR_BOX_COLS = [
    f'{_L_BOX}.box.center.x',
    f'{_L_BOX}.box.center.y',
    f'{_L_BOX}.box.center.z',
    f'{_L_BOX}.box.size.x',
    f'{_L_BOX}.box.size.y',
    f'{_L_BOX}.box.size.z',
    f'{_L_BOX}.box.heading',
    f'{_L_BOX}.type',
]

lidar_box_df = tk._read_cached('lidar_box')
actual_cols  = set(lidar_box_df.columns)
missing      = [c for c in REQUIRED_LIDAR_BOX_COLS if c not in actual_cols]

if missing:
    print('\n\u26a0\ufe0f  COLUMN MISMATCH — the following expected columns were NOT found:')
    for c in missing:
        print(f'   missing: {c}')
    print('\nActual columns:')
    for c in sorted(actual_cols):
        print(f'   {c}')
    print('\nFix: update the column names in train.py > WaymoLiDARDataset.__getitem__')
    print('     and in the eval / BEV cells below to match the actual column names.')
    raise RuntimeError('Column name mismatch — see output above.')
else:
    print('\n\u2705  All required lidar_box columns found.')

# ── Verify lidar XYZI loading works ───────────────────────────────────────────
ts0      = tk.get_timestamps()[0]
pts_test = tk.load_lidar_points_xyzi(ts0)
total    = sum(len(p) for p in pts_test)
print(f'\u2705  load_lidar_points_xyzi: {total:,} points across {len(pts_test)} lasers')
assert pts_test[0].shape[1] == 4, 'Expected (N, 4) XYZI — got wrong shape!'

print('\nPre-flight passed. Ready to train.')


In [ ]:
# §4 — LiDAR frame preview (BEV) — optional but recommended before training
from modules.visualize import plot_bev

PREVIEW_SEG_IDX = 0
PREVIEW_FRAME   = 0

tk.assign_segment(segs[PREVIEW_SEG_IDX])
ts = tk.get_timestamps()[PREVIEW_FRAME]

pts_list = tk.load_lidar_points_xyzi(ts)
pts_xyz  = [p[:, :3] for p in pts_list]
boxes_df = tk.load_lidar_boxes(ts)

fig = plot_bev(pts_xyz, boxes_df)
fig.suptitle(f'LiDAR BEV — segment {PREVIEW_SEG_IDX}, frame {PREVIEW_FRAME}',
             fontsize=12)
plt.tight_layout()
plt.show()

total_pts = sum(len(p) for p in pts_list)
gt_counts = {0: 0, 1: 0, 2: 0}
for _, row in boxes_df.iterrows():
    t = int(row[f'{_L_BOX}.type'])
    if t in LIDAR_DET_CLASS_MAP:
        gt_counts[LIDAR_DET_CLASS_MAP[t]] += 1

print(f'Points       : {total_pts:,} (across {len(pts_list)} lasers)')
print(f'GT boxes     : {len(boxes_df)} total')
print(f'  vehicle    : {gt_counts[0]}')
print(f'  pedestrian : {gt_counts[1]}')
print(f'  cyclist    : {gt_counts[2]}')


In [ ]:
# §5 — Training
#
# Resumes automatically from DRIVE_DIR/checkpoints/latest.pt if it exists.
# Progress tracked in DRIVE_DIR/progress.json — safe to re-run after disconnect.
#
# Expected loss curve (T4, 20 segs, 3 epochs/seg):
#   seg  1: loss ~400-800  (random init, focal loss is high)
#   seg  5: loss ~50-150   (heatmap learning to suppress background)
#   seg 10: loss ~10-30    (box regression kicking in)
#   seg 20: loss ~3-8      (converging; BEV mAP ~15-25% at this stage)

cmd = [
    sys.executable, 'train.py',
    '--config',    CONFIG,
    '--drive-dir', DRIVE_DIR,
]

if TOTAL_SEGS_OVERRIDE is not None:
    cmd += ['--total-segs', str(TOTAL_SEGS_OVERRIDE)]
if BATCH_OVERRIDE is not None:
    cmd += ['--batch', str(BATCH_OVERRIDE)]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# §6 — Load checkpoint & run inference on one frame
import torch
from models import build_detector

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
latest_ckpt = Path(DRIVE_DIR) / 'checkpoints' / 'latest.pt'

if not latest_ckpt.exists():
    raise FileNotFoundError(
        f'No checkpoint at {latest_ckpt} — run the training cell first.'
    )

ckpt     = torch.load(latest_ckpt, map_location=device, weights_only=False)
nn_model = ckpt['model'].to(device).eval()
print(f'Loaded checkpoint (trained through segment {ckpt.get("seg", "?")}).'
      f'  Device: {device}')

with open(CONFIG) as f:
    active_cfg = yaml.safe_load(f)
detector    = build_detector(active_cfg)
CLASS_NAMES = active_cfg['model'].get('class_names',
              ['vehicle', 'pedestrian', 'cyclist'])

# ── Ensure tk/segs are initialised (safe to re-run this cell independently) ───
if 'tk' not in dir() or 'segs' not in dir():
    from modules.waymo_open_dataset import ToolKit, LIDAR_DET_CLASS_MAP, _L_BOX
    import numpy as np
    tk   = ToolKit(split=active_cfg['data']['split'])
    segs = tk.list_segments()

# ── Inference on one frame ─────────────────────────────────────────────────────
INFER_SEG_IDX = 0
INFER_FRAME   = 5

tk.assign_segment(segs[INFER_SEG_IDX])
ts_infer   = tk.get_timestamps()[INFER_FRAME]
pts_list_i = tk.load_lidar_points_xyzi(ts_infer)
points_i   = np.concatenate(pts_list_i, axis=0)

with torch.no_grad():
    dets = detector.predict(nn_model, points_i, active_cfg)

print(f'{len(dets)} detections (score \u2265 {active_cfg["eval"]["score_thresh"]}):')
for d in sorted(dets, key=lambda x: -x['score'])[:20]:
    b = d['box']
    print(f'  {CLASS_NAMES[d["label"]]:<12} score={d["score"]:.3f}  '
          f'cx={b[0]:+.1f} cy={b[1]:+.1f} cz={b[2]:+.2f}  '
          f'dx={b[3]:.2f} dy={b[4]:.2f} dz={b[5]:.2f}')


In [ ]:
# §7 — BEV visualisation — predictions (coloured) vs GT (white outlines)
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 12), facecolor='#111')
ax.set_facecolor('#111')

# Point cloud scatter (height coloured)
pts_xy = np.concatenate([p[:, :2] for p in pts_list_i], axis=0)
pts_z  = np.concatenate([p[:, 2]  for p in pts_list_i], axis=0)
sc = ax.scatter(pts_xy[:, 0], pts_xy[:, 1], c=pts_z,
                cmap='plasma', s=0.3, vmin=-3, vmax=3, alpha=0.5)

# GT boxes (white)
gt_df = tk.load_lidar_boxes(ts_infer)
for _, row in gt_df.iterrows():
    if int(row[f'{_L_BOX}.type']) not in LIDAR_DET_CLASS_MAP:
        continue
    cx  = float(row[f'{_L_BOX}.box.center.x'])
    cy  = float(row[f'{_L_BOX}.box.center.y'])
    dx  = float(row[f'{_L_BOX}.box.size.x'])
    dy  = float(row[f'{_L_BOX}.box.size.y'])
    ax.add_patch(plt.Rectangle((cx - dx/2, cy - dy/2), dx, dy,
                               linewidth=1.2, edgecolor='white', facecolor='none'))

# Predicted boxes (class-coloured, opacity \u221d score)
DET_COLORS = {'vehicle': '#FF6B6B', 'pedestrian': '#51CF66', 'cyclist': '#339AF0'}
for d in dets:
    name = CLASS_NAMES[d['label']]
    b    = d['box']
    ax.add_patch(plt.Rectangle(
        (b[0] - b[3]/2, b[1] - b[4]/2), b[3], b[4],
        linewidth=1.5, edgecolor=DET_COLORS.get(name, 'yellow'),
        facecolor='none', alpha=min(d['score'] * 2.5, 1.0)
    ))

vox = active_cfg['voxel']
ax.set_xlim(vox['x_range']); ax.set_ylim(vox['y_range']); ax.set_aspect('equal')
ax.set_xlabel('X (m)', color='white', fontsize=12)
ax.set_ylabel('Y (m)', color='white', fontsize=12)
ax.tick_params(colors='white')
ax.set_title(f'BEV 3-D detections — seg {INFER_SEG_IDX}, frame {INFER_FRAME}',
             color='white', fontsize=14)
ax.legend(handles=[
    mpatches.Patch(color='white',   label='GT'),
    mpatches.Patch(color='#FF6B6B', label='vehicle'),
    mpatches.Patch(color='#51CF66', label='pedestrian'),
    mpatches.Patch(color='#339AF0', label='cyclist'),
], loc='upper right', facecolor='#222', labelcolor='white', framealpha=0.8)
plt.colorbar(sc, ax=ax, label='height (m)', shrink=0.7)
plt.tight_layout(); plt.show()
print(f'GT: {len(gt_df)} boxes   Predictions: {len(dets)}')


In [ ]:
# §8 — Evaluation — BEV mAP (axis-aligned IoU proxy)
#
# IoU thresholds follow Waymo convention:
#   vehicle: 0.7   pedestrian: 0.5   cyclist: 0.5
#
# AABB IoU ignores heading — under-estimates recall for angled boxes.
# Use the official Waymo eval library for competition-accurate numbers.

IOU_THRESHOLDS = {0: 0.7, 1: 0.5, 2: 0.5}
EVAL_FRAMES    = active_cfg['eval'].get('segs', 5)
EVAL_SEG_IDX   = 0

def _bev_iou_aabb(b1, b2):
    ix = max(0.0, min(b1[0]+b1[2]/2, b2[0]+b2[2]/2) - max(b1[0]-b1[2]/2, b2[0]-b2[2]/2))
    iy = max(0.0, min(b1[1]+b1[3]/2, b2[1]+b2[3]/2) - max(b1[1]-b1[3]/2, b2[1]-b2[3]/2))
    inter = ix * iy
    union = b1[2]*b1[3] + b2[2]*b2[3] - inter
    return inter / union if union > 0 else 0.0

def _compute_ap(pairs, n_gt):
    if n_gt == 0 or not pairs: return float('nan')
    pairs_s = sorted(pairs, key=lambda x: -x[0])
    tp_c = fp_c = 0
    prec, rec = [], []
    for _, is_tp in pairs_s:
        if is_tp: tp_c += 1
        else:     fp_c += 1
        prec.append(tp_c / (tp_c + fp_c)); rec.append(tp_c / n_gt)
    return sum(max((p for p, r in zip(prec, rec) if r >= t), default=0.0)
               for t in np.arange(0.0, 1.1, 0.1)) / 11.0

tk.assign_segment(segs[EVAL_SEG_IDX])
eval_ts   = tk.get_timestamps()[:EVAL_FRAMES]
all_preds = {c: [] for c in range(3)}
all_n_gt  = {c: 0  for c in range(3)}

print(f'Evaluating {len(eval_ts)} frames from segment {EVAL_SEG_IDX}...')
for ts_e in eval_ts:
    pts_e_list = tk.load_lidar_points_xyzi(ts_e)
    if not pts_e_list: continue
    pts_e = np.concatenate(pts_e_list, axis=0)
    with torch.no_grad():
        dets_e = detector.predict(nn_model, pts_e, active_cfg)

    gt_by_c = {c: [] for c in range(3)}
    for _, row in tk.load_lidar_boxes(ts_e).iterrows():
        t = int(row[f'{_L_BOX}.type'])
        if t not in LIDAR_DET_CLASS_MAP: continue
        c = LIDAR_DET_CLASS_MAP[t]
        gt_by_c[c].append([row[f'{_L_BOX}.box.center.x'], row[f'{_L_BOX}.box.center.y'],
                            row[f'{_L_BOX}.box.size.x'],  row[f'{_L_BOX}.box.size.y']])
        all_n_gt[c] += 1

    for c in range(3):
        matched = [False] * len(gt_by_c[c])
        for d in sorted([d for d in dets_e if d['label'] == c], key=lambda x: -x['score']):
            bp = [d['box'][0], d['box'][1], d['box'][3], d['box'][4]]
            best, bj = 0.0, -1
            for j, gt in enumerate(gt_by_c[c]):
                iou = _bev_iou_aabb(bp, gt)
                if iou > best: best, bj = iou, j
            if best >= IOU_THRESHOLDS[c] and bj >= 0 and not matched[bj]:
                matched[bj] = True; all_preds[c].append((d['score'], 1))
            else:
                all_preds[c].append((d['score'], 0))

print()
print(f'{"Class":<14} {"GT":>6} {"Dets":>6} {"IoU":>6} {"AP (BEV-AABB)":>15}')
print('-' * 52)
aps = []
for c, name in enumerate(CLASS_NAMES):
    ap = _compute_ap(all_preds[c], all_n_gt[c]); aps.append(ap)
    print(f'{name:<14} {all_n_gt[c]:>6} {len(all_preds[c]):>6} '
          f'{IOU_THRESHOLDS[c]:>6.1f} {ap:>15.3f}')
print(f'\nmAP (BEV-AABB) : {float(np.nanmean(aps)):.3f}')
print('\nNote: AABB IoU ignores heading. '
      'True Waymo 3D mAP requires rotated BEV IoU.')
